In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# ============================================================
# 0. Config
# ============================================================
BASE_DIR = Path("./")
RESULT_DIR = BASE_DIR / "result"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

EVENTS_FILE = BASE_DIR / "events_window0.csv"
QUANT_FILE = BASE_DIR / "event_quantification_window0_summary.csv"
PERSIST_FILE = BASE_DIR / "event_persistence_window0.csv"

K_CANDIDATES = [2, 3, 4]

# ============================================================
# 1. Load
# ============================================================
print("[1/6] Loading files...")

events = pd.read_csv(EVENTS_FILE)
quant = pd.read_csv(QUANT_FILE)
persist = pd.read_csv(PERSIST_FILE)

for df_ in [events, quant, persist]:
    df_["Date"] = pd.to_datetime(df_["Date"])

# ============================================================
# 2. Merge + feature table
# ============================================================
print("[2/6] Building feature table...")

df = (
    events[["Date"]]
    .merge(
        quant[["Date", "Magnitude"]],
        on="Date",
        how="left"
    )
    .merge(
        persist[["Date", "Peak", "PeakHorizon", "HalfLife", "AUC"]],
        on="Date",
        how="left"
    )
    .sort_values("Date")
    .reset_index(drop=True)
)

feature_cols = ["Magnitude", "Peak", "PeakHorizon", "HalfLife", "AUC"]

before_n = len(df)
df = df.dropna(subset=feature_cols).copy()
after_n = len(df)

print(f"[INFO] Events before dropna: {before_n}")
print(f"[INFO] Events after  dropna: {after_n}")

# ============================================================
# 3. Transform + scale
# ============================================================
print("[3/6] Transforming features...")

# clustering용 변환
df["Magnitude_log"] = np.log1p(df["Magnitude"])
df["AUC_log"] = np.log1p(df["AUC"])

cluster_cols = ["Magnitude_log", "Peak", "PeakHorizon", "HalfLife", "AUC_log"]

X = df[cluster_cols].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ============================================================
# 4. Select best k
# ============================================================
print("[4/6] Selecting cluster count...")

selection_rows = []

for k in K_CANDIDATES:
    model = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = model.fit_predict(X_scaled)

    sil = silhouette_score(X_scaled, labels)
    counts = pd.Series(labels).value_counts().sort_index()

    selection_rows.append({
        "k": k,
        "silhouette": round(float(sil), 6),
        "min_cluster_size": int(counts.min()),
        "cluster_sizes": " / ".join([f"C{idx}:{cnt}" for idx, cnt in counts.items()])
    })

selection_df = pd.DataFrame(selection_rows)

# 우선순위:
# 1) silhouette 최대
# 2) min_cluster_size 큰 것 우선
# 3) 같은 수준이면 더 작은 k 우선
selection_df = selection_df.sort_values(
    by=["silhouette", "min_cluster_size", "k"],
    ascending=[False, False, True]
).reset_index(drop=True)

best_k = int(selection_df.iloc[0]["k"])
print(f"[INFO] Selected k = {best_k}")

# ============================================================
# 5. Final clustering + naming
# ============================================================
print("[5/6] Fitting final model...")

final_model = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
labels = final_model.fit_predict(X_scaled)

df["Cluster"] = labels

summary = (
    df.groupby("Cluster")[feature_cols]
    .mean()
    .reset_index()
)

global_means = summary[feature_cols].mean()

def assign_label(row, global_means):
    mag_high = row["Magnitude"] >= global_means["Magnitude"]
    peak_high = row["Peak"] >= global_means["Peak"]
    auc_high = row["AUC"] >= global_means["AUC"]
    hl_high = row["HalfLife"] >= global_means["HalfLife"]
    ph_high = row["PeakHorizon"] >= global_means["PeakHorizon"]

    if mag_high and peak_high and auc_high and hl_high:
        return "Structural Persistent"
    elif mag_high and peak_high and (not hl_high):
        return "Transient Shock"
    elif ph_high and (not mag_high):
        return "Delayed Response"
    else:
        return "Minor / Local"

summary["ClusterLabel"] = summary.apply(assign_label, axis=1, global_means=global_means)

label_map = dict(zip(summary["Cluster"], summary["ClusterLabel"]))
df["ClusterLabel"] = df["Cluster"].map(label_map)

summary = summary[
    ["Cluster", "ClusterLabel", "Magnitude", "Peak", "PeakHorizon", "HalfLife", "AUC"]
].sort_values("Cluster").reset_index(drop=True)

result_df = df[
    ["Date", "Magnitude", "Peak", "PeakHorizon", "HalfLife", "AUC", "Cluster", "ClusterLabel"]
].copy()
result_df["Date"] = result_df["Date"].dt.strftime("%Y-%m-%d")

# ============================================================
# 6. Save outputs + boxplots
# ============================================================
print("[6/6] Saving outputs...")

result_df.to_csv(RESULT_DIR / "event_clusters_final.csv", index=False, encoding="utf-8-sig")
summary.to_csv(RESULT_DIR / "cluster_summary_final.csv", index=False, encoding="utf-8-sig")
selection_df.to_csv(RESULT_DIR / "model_selection_final.csv", index=False, encoding="utf-8-sig")

# ----------------------------
# Boxplots by cluster
# ----------------------------
plot_df = df.copy()
plot_features = ["Magnitude", "Peak", "PeakHorizon", "HalfLife", "AUC"]

fig, axes = plt.subplots(3, 2, figsize=(13, 12))
axes = axes.flatten()

cluster_order = sorted(plot_df["Cluster"].unique())
cluster_names = {
    c: f"C{c}\n{plot_df.loc[plot_df['Cluster'] == c, 'ClusterLabel'].iloc[0]}"
    for c in cluster_order
}

for i, col in enumerate(plot_features):
    ax = axes[i]

    data = [
        plot_df.loc[plot_df["Cluster"] == c, col].dropna().values
        for c in cluster_order
    ]

    ax.boxplot(
        data,
        labels=[cluster_names[c] for c in cluster_order],
        showfliers=True
    )
    ax.set_title(col)
    ax.set_xlabel("Cluster")
    ax.set_ylabel(col)
    ax.tick_params(axis="x", rotation=0)

# 마지막 빈 축 제거
if len(plot_features) < len(axes):
    for j in range(len(plot_features), len(axes)):
        fig.delaxes(axes[j])

fig.suptitle("Feature Distributions by Cluster", fontsize=16)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(RESULT_DIR / "feature_boxplots_by_cluster.png", dpi=220, bbox_inches="tight")
plt.close()

# ----------------------------
# Summary text
# ----------------------------
with open(RESULT_DIR / "run_summary_final.txt", "w", encoding="utf-8") as f:
    f.write("Event Clustering Final Summary\n")
    f.write("=" * 60 + "\n")
    f.write(f"Input files:\n")
    f.write(f"- {EVENTS_FILE}\n")
    f.write(f"- {QUANT_FILE}\n")
    f.write(f"- {PERSIST_FILE}\n\n")

    f.write(f"Events before dropna: {before_n}\n")
    f.write(f"Events after  dropna: {after_n}\n\n")

    f.write("Used features:\n")
    f.write("- Magnitude\n")
    f.write("- Peak\n")
    f.write("- PeakHorizon\n")
    f.write("- HalfLife\n")
    f.write("- AUC\n\n")

    f.write("Transform:\n")
    f.write("- Magnitude -> log1p (for clustering only)\n")
    f.write("- AUC -> log1p (for clustering only)\n")
    f.write("- all clustering variables -> z-score standardization\n\n")

    f.write("Model:\n")
    f.write("- AgglomerativeClustering(linkage='ward')\n")
    f.write(f"- k candidates: {K_CANDIDATES}\n")
    f.write(f"- selected k: {best_k}\n\n")

    f.write("Cluster labels:\n")
    for _, row in summary.iterrows():
        f.write(
            f"- Cluster {int(row['Cluster'])}: {row['ClusterLabel']} "
            f"(Magnitude={row['Magnitude']:.3f}, Peak={row['Peak']:.3f}, "
            f"PeakHorizon={row['PeakHorizon']:.3f}, HalfLife={row['HalfLife']:.3f}, "
            f"AUC={row['AUC']:.3f})\n"
        )

print("\n[DONE] Outputs saved in:")
print(RESULT_DIR.resolve())

print("\n[FILES]")
print("- event_clusters_final.csv")
print("- cluster_summary_final.csv")
print("- model_selection_final.csv")
print("- feature_boxplots_by_cluster.png")
print("- run_summary_final.txt")

[1/6] Loading files...
[2/6] Building feature table...
[INFO] Events before dropna: 52
[INFO] Events after  dropna: 38
[3/6] Transforming features...
[4/6] Selecting cluster count...
[INFO] Selected k = 4
[5/6] Fitting final model...
[6/6] Saving outputs...

[DONE] Outputs saved in:
D:\University\3-2.5\PADA_Lab\DC_and_meterials_2\12_Event_classification\result

[FILES]
- event_clusters_final.csv
- cluster_summary_final.csv
- model_selection_final.csv
- feature_boxplots_by_cluster.png
- run_summary_final.txt
